In this tutorial, we are going to evaluate the performance of the naive RAG and the GraphRAG algorithm on a [multi-hop RAG task](https://github.com/yixuantt/MultiHop-RAG).

## Setup
Make sure you install the necessary dependencies by running the following commands:

Import the necessary libraries, and set up your openai api key if needed:

In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [2]:
#os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"
import json
import sys
sys.path.append("../..")

import nest_asyncio
nest_asyncio.apply()
import logging

logging.basicConfig(level=logging.WARNING)
logging.getLogger("nano-graphrag").setLevel(logging.INFO)
from nano_graphrag import GraphRAG, QueryParam
from datasets import Dataset 
from ragas import evaluate
from ragas.metrics import (
    answer_correctness,
    answer_similarity,
    answer_relevancy
)

c:\Github\nano-graphrag\.conda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Download the dataset from [Github Repo](https://github.com/yixuantt/MultiHop-RAG/tree/main/dataset). 
If should contain two files:
- `MultiHopRAG.json`
- `corpus.json`

After downloading the dataset, replace the below paths to the paths on your machine.

In [3]:

multi_hop_rag_file = "./fixtures/MultiHopRAG.json"
multi_hop_corpus_file = "./fixtures/corpus.json"

## Preprocess

In [4]:

with open(multi_hop_rag_file) as f:
    multi_hop_rag_dataset = json.load(f)
with open(multi_hop_corpus_file) as f:
    multi_hop_corpus = json.load(f)

corups_url_refernces = {}
for cor in multi_hop_corpus:
    corups_url_refernces[cor['url']] = cor

We only use the top-100 queries for evaluation.

In [5]:
multi_hop_rag_dataset = multi_hop_rag_dataset[:100]
print("Queries have types:", set([q['question_type'] for q in multi_hop_rag_dataset]))
total_urls = set()
for q in multi_hop_rag_dataset:
    total_urls.update([up['url'] for up in q['evidence_list']])
corups_url_refernces = {k:v for k, v in corups_url_refernces.items() if k in total_urls}

total_corpus = [f"## {cor['title']}\nAuthor: {cor['author']}, {cor['source']}\nCategory: {cor['category']}\nPublised: {cor['published_at']}\n{cor['body']}" for cor in corups_url_refernces.values()]

print(f"We will need {len(total_corpus)} articles:")
print(total_corpus[0][:200], "...")

Queries have types: {'temporal_query', 'null_query', 'comparison_query', 'inference_query'}
We will need 139 articles:
## ASX set to drop as Wall Street’s September slump deepens
Author: Stan Choe, The Sydney Morning Herald
Category: business
Publised: 2023-09-26T19:11:30+00:00
ETF provider Betashares, which manages $ ...


Add index for the `total_corups` using naive RAG and GraphRAG

In [6]:
import ollama
import numpy as np
from nano_graphrag._utils import compute_args_hash, wrap_embedding_func_with_attrs

# Ollama settings
MODEL = "qwen2.5:14b"  # or any other model you have in Ollama
EMBEDDING_MODEL = "qwen2.5:14b"
EMBEDDING_MODEL_DIM = 5120
EMBEDDING_MODEL_MAX_TOKENS = 16384

async def ollama_model_if_cache(
    prompt, system_prompt=None, history_messages=[], **kwargs
) -> str:
    kwargs.pop("max_tokens", None)
    kwargs.pop("response_format", None)

    ollama_client = ollama.AsyncClient()
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    hashing_kv = kwargs.pop("hashing_kv", None)
    messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})
    
    if hashing_kv is not None:
        args_hash = compute_args_hash(MODEL, messages)
        if_cache_return = await hashing_kv.get_by_id(args_hash)
        if if_cache_return is not None:
            return if_cache_return["return"]
            
    response = await ollama_client.chat(model=MODEL, messages=messages, **kwargs)
    result = response["message"]["content"]
    #result = result.split("</think>")[-1] # 去掉<think>
    if hashing_kv is not None:
        await hashing_kv.upsert({args_hash: {"return": result, "model": MODEL}})
    return result

@wrap_embedding_func_with_attrs(
    embedding_dim=EMBEDDING_MODEL_DIM,
    max_token_size=EMBEDDING_MODEL_MAX_TOKENS,
)
async def ollama_embedding(texts: list[str]) -> np.ndarray:
    embed_text = []
    for text in texts:
        data = ollama.embeddings(model=EMBEDDING_MODEL, prompt=text)
        embedding = np.array(data["embedding"])
        # Ensure the embedding dimension matches
        assert embedding.shape[0] == EMBEDDING_MODEL_DIM, f"Expected dimension {EMBEDDING_MODEL_DIM}, got {embedding.shape[0]}"
        embed_text.append(embedding)
    return np.vstack(embed_text)



In [7]:
# First time indexing will cost many time, roughly 15~20 minutes
from nano_graphrag._llm import openai_complete_if_cache  # 添加这行
from typing import Optional, List
from nano_graphrag._utils import compute_args_hash

graphrag_func = GraphRAG(
    working_dir="nano_graphrag_cache_multihop_rag_test_qwen25_14b",
    enable_naive_rag=True,
    embedding_func_max_async=4,
    embedding_batch_num=64,
    best_model_func=ollama_model_if_cache,
    cheap_model_func=ollama_model_if_cache,
    embedding_func=ollama_embedding
)

graphrag_func.insert(total_corpus)

INFO:nano-graphrag:Creating working directory nano_graphrag_cache_multihop_rag_test_qwen25_14b
INFO:nano-graphrag:Load KV full_docs with 0 data
INFO:nano-graphrag:Load KV text_chunks with 0 data
INFO:nano-graphrag:Load KV llm_response_cache with 0 data
INFO:nano-graphrag:Load KV community_reports with 0 data
INFO:nano-graphrag:[New Docs] inserting 139 docs
INFO:nano-graphrag:[New Chunks] inserting 408 chunks
INFO:nano-graphrag:Insert chunks for naive RAG
INFO:nano-graphrag:Inserting 408 vectors to chunks
INFO:nano-graphrag:[Entity Extraction]...


INFO:nano-graphrag:Inserting 407 vectors to entities


INFO:nano-graphrag:[Community Report]...
INFO:nano-graphrag:Each level has communities: {0: 4}
INFO:nano-graphrag:Generating by levels: [0]
INFO:nano-graphrag:JSON data successfully extracted.


INFO:nano-graphrag:JSON data successfully extracted.


INFO:nano-graphrag:JSON data successfully extracted.


INFO:nano-graphrag:JSON data successfully extracted.


INFO:nano-graphrag:Writing graph with 441 nodes, 180 edges


Look at the response of different RAG methods on the first query:

In [8]:
response_formate = "Single phrase or sentence, concise and no redundant explanation needed. If you don't have the answer in context, Just response 'Insufficient information'"
naive_rag_query_param = QueryParam(mode='naive', response_type=response_formate)
naive_rag_query_only_context_param = QueryParam(mode='naive', only_need_context=True)
local_graphrag_query_param = QueryParam(mode='local', response_type=response_formate)
local_graphrag_only_context__param = QueryParam(mode='local', only_need_context=True)
global_graphrag_query_param = QueryParam(mode='global', response_type=response_formate)
global_graphrag_only_context__param = QueryParam(mode='global', only_need_context=True)

In [9]:
query = multi_hop_rag_dataset[0]
print("Question:", query['query'])
print("GroundTruth Answer:", query['answer'])

Question: Who is the individual associated with the cryptocurrency industry facing a criminal trial on fraud and conspiracy charges, as reported by both The Verge and TechCrunch, and is accused by prosecutors of committing fraud for personal gain?
GroundTruth Answer: Sam Bankman-Fried


In [10]:
print("NaiveRAG Answer:", graphrag_func.query(query['query'], param=naive_rag_query_param))

INFO:nano-graphrag:Truncate 20 to 20 chunks


NaiveRAG Answer: The individual you are referring to is likely Alexander Vinnik, though based on your description and the popular high-profile cases, it seems like you might be thinking about one specific case often highlighted in media. However, a very well-known person facing similar charges in relation to cryptocurrency fraud as per multiple reports including The Verge and TechCrunch would be Alexei Vaney (though most prominent recent cases highlight Alexandr Vinnik). But the most famous case fitting your description exactly is probably related to Sam Bankman-Fried (SBF), founder of FTX, but he was more associated with securities fraud rather than initial reports on crypto conspiracy and early fraudulent exchanges.

If we're pinpointing specifically someone involved in a major reported fraud case involving early cryptocurrency activities:
**Alexander Vinnik** fits the specific description given. He is the founder of Bitcoin exchange BTC-e who faced charges related to money launderin

In [11]:
print("Local GraphRAG Answer:", graphrag_func.query(query['query'], param=local_graphrag_query_param))

INFO:nano-graphrag:Using 20 entites, 0 communities, 13 relations, 12 text units


Local GraphRAG Answer: The individual you are referring to is likely Alex van de Sande, but if we consider one of the most high-profile cases in the cryptocurrency industry related to your description, it could be tied to individuals like Jerome "Jered" Kent Kahan or others depending on the specific timeframe and context. However, a major case that matches closely with the details you've provided is likely referring to someone like Evro Smailbegovic.

But if we consider one of the most notable cases involving high-profile figures in cryptocurrency accused of fraud and conspiracy, it's often associated with individuals such as Alex van de Sande who was part of the Centra Tech case or other similar fraudulent ICO schemes. However, a prominent figure facing charges that aligns closely to your description could be co-founders of certain crypto ventures like Telegram and others related to SEC lawsuits.

If you're referring to a very specific case recently covered by The Verge and TechCrunch

In [12]:
print("Global GraphRAG Answer:", graphrag_func.query(query['query'], param=global_graphrag_query_param))

INFO:nano-graphrag:Revtrieved 4 communities
INFO:nano-graphrag:Grouping to 1 groups for global search
INFO:nano-graphrag:JSON data successfully extracted.


Global GraphRAG Answer: Sorry, I'm not able to provide an answer to that question.


Great! Now we're ready to evaluate more detailed metrics. We will use [ragas](https://docs.ragas.io/en/stable/) to evalue the answers' quality.

In [13]:
questions = [q['query'] for q in multi_hop_rag_dataset]
labels = [q['answer'] for q in multi_hop_rag_dataset]

In [14]:
from tqdm import tqdm
logging.getLogger("nano-graphrag").setLevel(logging.WARNING)

naive_rag_answers = [
    graphrag_func.query(q, param=naive_rag_query_param) for q in tqdm(questions)
]

100%|██████████| 100/100 [10:37<00:00,  6.38s/it]


In [15]:
local_graphrag_answers = [
    graphrag_func.query(q, param=local_graphrag_query_param) for q in tqdm(questions)
]

100%|██████████| 100/100 [11:04<00:00,  6.65s/it]


In [16]:
global_graphrag_answers = [
    graphrag_func.query(q, param=global_graphrag_query_param) for q in tqdm(questions)
]

 31%|███       | 31/100 [00:51<01:55,  1.68s/it]ERROR:nano-graphrag:JSON decoding failed: Expecting value: line 1 column 292 (char 291). Attempted string: {
    "points": [
        {"description": "There i...
100%|██████████| 100/100 [03:19<00:00,  2.00s/it]


In [17]:
from ragas.llms import LangchainLLMWrapper
from langchain.llms import Ollama
from langchain.callbacks.manager import CallbackManager
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# Configure Ollama with increased timeout and temperature
ollama_llm = Ollama(model="qwen2.5:14b")

# Configure the LLM wrapper with custom parameters
llm = LangchainLLMWrapper(ollama_llm)

# Update the metrics configuration
answer_correctness.llm = llm
answer_similarity.llm = llm

#answer_relevancy.llm = llm
#answer_relevancy.embeddings = embedding

# Modify evaluation to use smaller batches
naive_results = evaluate(
    Dataset.from_dict({
        "question": questions,
        "ground_truth": labels,
        "answer": naive_rag_answers,
    }),
    metrics=[
        #answer_relevancy, # 有bug
        answer_correctness,
        answer_similarity,
    ]
)

C:\Users\15950\AppData\Local\Temp\ipykernel_39820\2961884224.py:8: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaLLM``.
  ollama_llm = Ollama(model="qwen2.5:14b")
Evaluating: 100%|██████████| 200/200 [12:07<00:00,  3.64s/it]


In [18]:
local_graphrag_results = evaluate(
    Dataset.from_dict({
        "question": questions,
        "ground_truth": labels,
        "answer": local_graphrag_answers,
    }),
    metrics=[
        #answer_relevancy,
        answer_correctness,
        answer_similarity,
    ],
)

Evaluating: 100%|██████████| 200/200 [12:35<00:00,  3.78s/it]


In [19]:
global_graphrag_results = evaluate(
    Dataset.from_dict({
        "question": questions,
        "ground_truth": labels,
        "answer": global_graphrag_answers,
    }),
    metrics=[
        #answer_relevancy,
        answer_correctness,
        answer_similarity,
    ],
)

Evaluating: 100%|██████████| 200/200 [06:09<00:00,  1.85s/it]


In [20]:
print("Naive RAG results", naive_results)
print("Local GraphRAG results", local_graphrag_results)
print("global GraphRAG results", global_graphrag_results)

Naive RAG results {'answer_correctness': 0.5413, 'semantic_similarity': 0.7529}
Local GraphRAG results {'answer_correctness': 0.5381, 'semantic_similarity': 0.7521}
global GraphRAG results {'answer_correctness': 0.3467, 'semantic_similarity': 0.7760}
